## Router Multi-Agent Design Pattern with LangChain

![router_workflow](./Images/router_workflow.png)

### Installing Utilities and Libraries

In [1]:
%pip install langchain-anthropic==1.5.4 anthropic==0.120.2 python-dotenv==1.2.2

Looking in indexes: https://pypi.org/simple/

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: /usr/local/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Setting up the Environment

In [2]:
import os
from dotenv import load_dotenv
from typing import Literal, TypedDict
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command

load_dotenv()
anthropic_api_key = os.getenv("CLAUDE_API_KEY")
anthropic_model_name = os.getenv("CLAUDE_MODEL_NAME")

### Instantiate the ChatAnthropic Class

In [3]:
from langchain_anthropic import ChatAnthropic

model = ChatAnthropic(
    model_name = anthropic_model_name,
    api_key = anthropic_api_key
)

### Define the State

In [5]:
class RouterState(TypedDict):
    query: str
    final_answer: str

### Create Specialized Agents

In [6]:
sales_agent = create_agent(
    model=model,
    tools=[],
    system_prompt=(
        "You are a sales specialist. "
        "Answer questions about products, pricing, "
        "plans, purchases, and upgrades."
    ),
)


support_agent = create_agent(
    model=model,
    tools=[],
    system_prompt=(
        "You are a technical support specialist. "
        "Help users troubleshoot technical problems, "
        "login issues, errors, and product usage."
    ),
)


billing_agent = create_agent(
    model=model,
    tools=[],
    system_prompt=(
        "You are a billing specialist. "
        "Answer questions about invoices, payments, "
        "refunds, and billing issues."
    ),
)

### Create the Router

In [7]:
def route_query(
    state: RouterState,
) -> Command[
    Literal[
        "sales_agent",
        "support_agent",
        "billing_agent",
    ]
]:

    query = state["query"].lower()

    if any(
        word in query
        for word in [
            "price",
            "pricing",
            "purchase",
            "buy",
            "upgrade",
            "plan",
        ]
    ):
        selected_agent = "sales_agent"

    elif any(
        word in query
        for word in [
            "invoice",
            "payment",
            "refund",
            "charged",
            "billing",
        ]
    ):
        selected_agent = "billing_agent"

    else:
        selected_agent = "support_agent"

    print(f"Router → {selected_agent}")

    return Command(
        goto=selected_agent
    )

### Create the Agent Nodes for Execution

In [8]:
def call_sales_agent(state: RouterState):

    result = sales_agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content=state["query"]
                )
            ]
        }
    )

    return {
        "final_answer":
            result["messages"][-1].content
    }


def call_support_agent(state: RouterState):

    result = support_agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content=state["query"]
                )
            ]
        }
    )

    return {
        "final_answer":
            result["messages"][-1].content
    }


def call_billing_agent(state: RouterState):

    result = billing_agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content=state["query"]
                )
            ]
        }
    )

    return {
        "final_answer":
            result["messages"][-1].content
    }

### Build the Router Graph

In [9]:
builder = StateGraph(RouterState)

builder.add_node(
    "router",
    route_query,
)

builder.add_node(
    "sales_agent",
    call_sales_agent,
)

builder.add_node(
    "support_agent",
    call_support_agent,
)

builder.add_node(
    "billing_agent",
    call_billing_agent,
)


builder.add_edge(
    START,
    "router",
)

builder.add_edge(
    "sales_agent",
    END,
)

builder.add_edge(
    "support_agent",
    END,
)

builder.add_edge(
    "billing_agent",
    END,
)


graph = builder.compile()

### Generate the Mermaid Diagram of the Workflow

In [10]:
png = graph.get_graph().draw_mermaid_png()

with open("router_workflow.png", "wb") as f:
    f.write(png)

### Invoke the Workflow

Queries to try:

1) "I was charged twice for my subscription."

2) "How much does the premium plan cost?"
"""

In [11]:
result = graph.invoke(
    {
        "query":
             "I was charged twice for my subscription"
    }
)

print("\nFinal Answer:")
print(result["final_answer"])

Router → billing_agent

Final Answer:
[{'signature': 'EpUDCpABCBIYAipAYDZjd27/1AKcq8seGd4OUGapfFJ3cyEoIxpWSxh6wLn4Z0oD7xE/ZeHDTw4+LRrU2NsNs4GZDsxUPPtLcpKE4TIPY2xhdWRlLXNvbm5ldC01OABCCHRoaW5raW5nWiQ4YTdhNWVmNC1iYjMxLTQwYTItYTA0My02NjVhYjJkZGU4NDaoAYSQ39UGEgxPifr0BRSWC/W/QQUaDC49KkxSqBTrBFdRNCIwUNn/5hZPVNE2/YSmEDm+r6Gwv+UdcHlMQPviEcHv+Q+GkxmfnqopqH6AfNjfzRZLKrEBonJRDr183b8FiLvnrE3L5nWGjSwnqoHBMO8kecQZ7jqs6ZKDPTPJQPGatgJODoIIP5TeiUQz49C5EAfT46G3rmKofH0jH+DMbTIXV1ISZEaTaGTNMfpugW+DcHnSojgTCxUnJAxx1HCThcrdqlLeUEaxwTc14Ez9a9UsDQzUwPSFPk0to2YEk5eks72NsUZEslUFqlCd8FM57eCKSNe3/3c4gHKboN4nH/huoLUXnD+PGAE=', 'thinking': '', 'type': 'thinking'}, {'text': "I'm sorry to hear about the duplicate charge — let's get that sorted out. To look into this properly, could you provide a few details?\n\n1. **Account/Email** associated with your subscription\n2. **Approximate dates** of the two charges you noticed\n3. **Amount charged** each time (to confirm they're identical or if amounts differ)\n4. **Payment